# 19. 形状、合并与拆分

<!-- module-learning-arc:start -->
> **NumPy 模块主线｜第 4 / 6 步：调整并组合分析结构**
>
> **持续应用背景：** 为区域仓库建立补货预警矩阵：把门店、商品、库存和需求组织成数组，逐步完成定位、广播计算、排序和抽样复核。
>
> **承接上一阶段：** 索引、切片与筛选  →  **本章任务：** 形状、合并与拆分  →  **下一步：** 向量化与广播
>
> **大作业连接：** 本章练习将成为《连锁门店补货预警矩阵》的一部分，最终需要把数组建模、风险筛选、广播计算和抽样复核组合成一份可执行的补货清单。
<!-- module-learning-arc:end -->


## 本章场景

真实数据很少天生就是方方正正一张表——统计分析可能按“天数”排成一行行，建模时又要按“行×特征”重新排成矩阵。



## 本章目标

学完本章，你将能够：

- **理解**：理解 ndarray 形状、reshape 与 concatenate/stack 合并。
- **操作**：能reshape、拼接、拆分数组。
- **迁移**：能把多列经营数据重组/合并成适合计算的形状。


## 19.1 核心概念

**背景引入**：真实数据很少天生就是方方正正一张表——统计分析可能按“天数”排成一行行，建模时又要按“行×特征”重新排成矩阵。reshape 就是给数据换一份“衣柜样式”而不弄丢任何一件衣服；合并把不同来源的数拼到一起，拆分又把一整块切回若干小组。学完这一章，你就能随手把数组换成计算和可视化想要的样子，而不再卡在“形状不对”上。

- reshape前后元素总数必须一致。
- axis=0通常表示按行方向操作，axis=1表示按列方向操作。
- 合并前除目标轴外的其他维度必须匹配。

**口诀**：reshape 只重排不丢数，T 换行列视角，ravel 摊平回一列。


## 19.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| reshape() 与 transpose() | `np.arange()`、`matrix.reshape()`、`matrix.transpose()` | reshape改变形状，transpose交换维度，不改变元素总数。 | reshape目标元素数不一致 |
| concatenate() | `np.array()`、`np.concatenate()` | concatenate沿已有轴拼接，其他轴的形状必须兼容。 | 混淆按行和按列合并 |
| stack() | `np.array()`、`np.stack()` | stack会创建一个新的维度，适合把同形状数组组成批次。 | 使用split拆分不能整除的数组 |
| split() | `np.arange()`、`np.split()`、`part.tolist()` | split按指定位置把一个数组拆成多个数组。 | reshape目标元素数不一致 |
| hsplit() 与 vsplit() | `np.arange()`、`part.tolist()`、`np.hsplit()`、`np.vsplit()` | hsplit按列拆分，vsplit按行拆分。 | 混淆按行和按列合并 |


## 19.3 示例 1：形状变换与转置

reshape只改变视图结构，不改变元素顺序。

**背景引入**：记账数据通常是一长串数字（比如 1 到 12 的销售额），可汇报和建模需要的却是“3 行 4 列”这样的表格。形状不对，后面按行算、按列算就全乱套。reshape 就是给数据重排格子而不丢一个数，再配合转置换个视角看同一批数。

**讲解**：reshape 先数清元素总数再安排新形状，transpose / .T 交换行列维度，ravel 展平回一维；三者都只改“怎么摆放”，不改变元素本身。

- reshape(3, 4) 把 12 个元素重排成 3 行 4 列，元素总数必须一致；
- matrix.T 是转置：3×4 变 4×3，相当于把行、列互换，看同一批数的另一面；
- ravel() 把矩阵摊平回一维，元素顺序不变；
- **口诀**：reshape 重排格子、T 换行列视角、ravel 摊平回一列，摆法变、内容没变。


<!-- math-foundation:chapter-19 -->
### 数学推导｜变形必须保持元素总数

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜计算原数组容量。** 原形状 $(d_1,\ldots,d_r)$ 的元素数是

$$
N_{old}=\prod_{k=1}^{r}d_k
$$

**第 2 步｜计算目标容量。** 新形状 $(e_1,\ldots,e_s)$ 需要

$$
N_{new}=\prod_{j=1}^{s}e_j
$$

**第 3 步｜比较容量。** `reshape` 只重新解释同一串元素，不会创造或删除数据，所以必须有 $N_{old}=N_{new}$。

**把上面的关系收束为本章计算式：**

$$
\prod_{k=1}^{r}d_k=\prod_{j=1}^{s}e_j
$$

**符号解释：** $d_k$ 是原形状各维长度，$e_j$ 是目标形状各维长度。

**代码对应：** 执行 `reshape(new_shape)` 前检查 `np.prod(old_shape) == np.prod(new_shape)`。

**使用边界：** 元素数量相等只保证能够变形，不保证新的轴仍具有正确业务含义。


In [ ]:
import numpy as np

values = np.arange(1, 13)
matrix = values.reshape(3, 4)
print(matrix)
print("转置:\n", matrix.T)
print("展平:", matrix.ravel())


## 19.4 示例 2：数组合并

vstack按行叠加，hstack按列拼接。

**背景引入**：同一类数据常被拆成几份——上个月的销售、这个月的销售分开放。做汇总时要先把它们拼回一张大表，但拼法有两种：上下叠（加到行）还是左右拼（加到列）。方向用错，表的结构就完全不对。

**讲解**：vstack 沿行方向上下叠（行数增加），hstack 沿列方向左右拼（列数增加），stack 则新增一维把同形状数组组成批次。

- np.vstack([first, second])：把两个 2×2 按行叠成 4×2；
- np.hstack([first, second])：把两个 2×2 按列拼成 2×4；
- np.stack([first, second], axis=0) 不改原有行列，而是新增第 0 维（shape 变 2,2,2），适合把同类数组组“批次”；
- **口诀**：要行变多用 vstack，要列变长用 hstack，要组批次用 stack 新增一维。


In [ ]:
first = np.array([[1, 2], [3, 4]])
second = np.array([[5, 6], [7, 8]])
print("按行:\n", np.vstack([first, second]))
print("按列:\n", np.hstack([first, second]))
print("新增维度:\n", np.stack([first, second], axis=0).shape)


## 19.5 示例 3：数组拆分

split要求能够等分，array_split允许不等分。

**背景引入**：把一整年 12 个月的数据“切”成季度来看，是经营复盘很常见的动作。但当数据量不能被整除（比如 12 个切成 5 段）时，硬切就会报错。这时候需要一种能“多退少补”的切法。

**讲解**：split 要求能整除、切出的每段长度相同；array_split 遇到不能整除时会自动把多余的元素分配到靠前的段，各段长度基本均衡。

- np.split(monthly, 4)：把 12 个数等分成 4 个季度，每段 3 个；
- np.array_split(monthly, 5)：12 个切 5 段不能整除，前面几段多一个、后面少一个；
- 拆分返回的是多个子数组组成的列表，可用 len(part) 逐段检查长度；
- **口诀**：能整除用 split、等分整齐；不能整除用 array_split、余数自动摊。


In [ ]:
monthly = np.arange(1, 13)
quarters = np.split(monthly, 4)
uneven = np.array_split(monthly, 5)
print("季度:", quarters)
print("不等分长度:", [len(part) for part in uneven])


## 19.6 核心操作独立示例

下面每个代码单元格只演示一个核心方法、函数或语法操作。请先阅读方法名称和任务说明，再单独运行当前单元格；示例尽量自带最小输入，不要求依赖前一个单元格留下的变量。


In [ ]:
# reshape() 与 transpose()
# reshape改变形状，transpose交换维度，不改变元素总数。
import numpy as np

matrix = np.arange(6).reshape(2, 3)
print(matrix.reshape(3, 2))
print(matrix.transpose())


In [ ]:
# concatenate()
# concatenate沿已有轴拼接，其他轴的形状必须兼容。
import numpy as np

left = np.array([[1, 2], [3, 4]])
right = np.array([[5, 6], [7, 8]])
print(np.concatenate([left, right], axis=0))


In [ ]:
# stack()
# stack会创建一个新的维度，适合把同形状数组组成批次。
import numpy as np

first = np.array([1, 2, 3])
second = np.array([4, 5, 6])
print(np.stack([first, second], axis=0))


In [ ]:
# split()
# split按指定位置把一个数组拆成多个数组。
import numpy as np

values = np.arange(8)
parts = np.split(values, 4)
print([part.tolist() for part in parts])


In [ ]:
# hsplit() 与 vsplit()
# hsplit按列拆分，vsplit按行拆分。
import numpy as np

matrix = np.arange(12).reshape(3, 4)
print([part.tolist() for part in np.hsplit(matrix, 2)])
print([part.tolist() for part in np.vsplit(matrix, 3)])


**练一练 16.6**：把 1 到 12 排成一个 3×4 的数组，完成三件事：① 用 reshape 把它改回 4×3，保存为 reshaped（元素总数不变）；② 沿 axis=1 把它与自身合并成一个 3×8 的数组 merged；③ 用 split 把 merged 按 2 块等分，打印出第一块的形状。数据用简单的数值数组即可。


In [ ]:
# 请在下方填写代码
import numpy as np

# TODO ①: 用 reshape 把 base 改成 4×3，保存在 reshaped
# TODO ②: 沿 axis=1 把 base 与自身合并成 3×8，保存在 merged
# TODO ③: 用 split 把 merged 沿 axis=1 等分成 2 块，把第一块保存在 first_part 并打印其形状


In [ ]:
import numpy as np

base = np.arange(1, 13).reshape(3, 4)
reshaped = base.reshape(4, 3)  # ① reshape，元素总数 12 不变
merged = np.concatenate([base, base], axis=1)  # ② 沿列方向拼接成 3×8
parts = np.split(merged, 2, axis=1)  # ③ 沿列等分成 2 块
print(reshaped.shape, merged.shape, parts[0].shape)


**输出解读**：`reshaped` 是 4×3（`reshape(4, 3)`，12 个元素重排，总数不变）；`merged` 沿 `axis=1` 与自身拼接，变成 3×8；`np.split(merged, 2)` 把 3×8 均分成两块，第一块 shape 是 `(3, 4)`。要点：reshape 只改摆法不改数；合并方向决定"行变多还是列变多"；拆分要能整除才叫"等分"。


## 19.7 独立迁移练习

先预测 shape，再修改一个数组或筛选条件，解释结果变化。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 19.8 本章实训：axis与布尔筛选

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np

matrix = np.arange(1, 13).reshape(3, 4)
print("原数组：\n", matrix)
print("每行合计：", matrix.sum(axis=1))
print("每列合计：", matrix.sum(axis=0))


### 19.8.1 第一个结果怎么读

`axis=1` 保留行，沿列方向计算；`axis=0` 保留列，沿行方向计算。先看 shape，再解释结果长度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
even = matrix[matrix % 2 == 0]
print("偶数：", even)
print("偶数数量：", even.size)
print("偶数平均值：", even.mean())


### 19.8.2 第二个结果怎么读

第二个实验不改原数组，而是用布尔条件筛选新数组。请思考：如果条件改成 `matrix > 8`，输出会怎样变化？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 19.9 错误恢复：数组形状不匹配怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import numpy as np

matrix = np.arange(6).reshape(2, 3)
try:
    result = matrix + np.array([10, 20])
except ValueError as error:
    print("形状问题：", type(error).__name__)
    result = matrix + np.array([10, 20, 30])
print("修复后的结果：")
print(result)


### 19.9.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

先看两个数组的 shape，再判断能否广播。修复不是随意 reshape，而是让数据结构和业务含义一致。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 19.10 易错点提醒

- reshape目标元素数不一致
- 混淆按行和按列合并
- 使用split拆分不能整除的数组


## 19.11 练习与作业

1. 把1到24变成4×6数组
2. 拆成上半部和下半部
3. 转置后输出形状

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 19.12 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“把1到24变成4×6数组”。
2. **独立完成**：不复制示例代码，完成“拆成上半部和下半部”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“转置后输出形状”，用一两句话说明你修改了什么。

### 19.12.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 19.12.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
import numpy as np

# TODO: 创建4×6数组
# TODO: 拆成上半部和下半部
# TODO: 转置
# TODO：请在下方完成 —— 16.12 练习与作业 1. 把1到24变成4×6数组 2. 拆成上半部和下半部 3. 转置后输出形状 提交前检查：代码


In [ ]:
import numpy as np

matrix = np.arange(1, 25).reshape(4, 6)
top, bottom = np.split(matrix, 2, axis=0)
transposed = matrix.T
print("原形状:", matrix.shape)
print("上半部:\n", top)
print("下半部:\n", bottom)
print("转置形状:", transposed.shape)


## 19.13 小结

使用reshape、转置、合并和拆分调整数组结构。

**迁移思考**：

1. 如果有一个 12 元素数组，除了 reshape(3, 4) 还可以变成什么形状？
2. 为什么 vstack 和 hstack 要求除目标轴外的其他维度必须匹配？



### 19.13.1 你已经掌握

- 改变数组形状
- 区分展平方法
- 按不同轴合并
- 将数组拆分为多个部分


### 19.13.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 19.13.3 需要注意

- reshape目标元素数不一致
- 混淆按行和按列合并
- 使用split拆分不能整除的数组


### 19.13.4 完成检查

- [ ] 能够改变数组形状
- [ ] 能够区分展平方法
- [ ] 能够按不同轴合并
- [ ] 能够将数组拆分为多个部分


### 19.13.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
